# Biohub - Cell Tracking During Development: Minimal Submission Pipeline

**Milestone 1 (no ML yet).** This notebook builds a clean, schema-valid dummy
`submission.csv` for every test `.zarr` dataset:

1. Finds all test `.zarr` folders under `/kaggle/input`.
2. Safely reads each Zarr volume's metadata (shape), with a fallback if a
   store is missing, corrupted, or has an unexpected layout.
3. Emits 3 node rows (`t=0,1,2` at the volume center) + 2 edges linking them,
   per dataset.
4. Validates the full submission against the competition's structural rules.
5. Saves to `/kaggle/working/submission.csv` and prints its shape/head.

Required columns: `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`


## 1. Imports & configuration

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Sequence

import pandas as pd

try:
    import zarr
except ImportError:
    zarr = None
    print("[warn] the 'zarr' package is not installed; metadata reads will "
          "use the fallback shape for every dataset.")

REQUIRED_COLUMNS = [
    "id", "dataset", "row_type", "node_id",
    "t", "z", "y", "x", "source_id", "target_id",
]

# Fallback (t, z, y, x) shape used when a volume's metadata can't be read.
DEFAULT_SHAPE = (3, 64, 64, 64)

CANDIDATE_INPUT_ROOTS = [Path("/kaggle/input")]


## 2. Find all test `.zarr` folders

Recursively scans the input root(s) for `*.zarr` directories. Prefers ones that live under a path component literally named `test`; if none do (e.g. the dataset ships test-only data with no explicit `test` folder), falls back to treating every discovered `.zarr` folder as test data.

In [ ]:
def find_test_zarr_dirs(roots: Sequence[Path] = CANDIDATE_INPUT_ROOTS) -> list[Path]:
    """Recursively locate every *.zarr folder belonging to the test split."""
    all_zarr_dirs: list[Path] = []
    for root in roots:
        try:
            if not root.exists():
                continue
            for path in root.rglob("*.zarr"):
                if path.is_dir():
                    all_zarr_dirs.append(path)
        except OSError as exc:
            print(f"[warn] could not scan {root}: {exc!r}")

    if not all_zarr_dirs:
        return []

    test_dirs = [p for p in all_zarr_dirs if "test" in {part.lower() for part in p.parts}]
    if test_dirs:
        return sorted(set(test_dirs))

    print(
        "[warn] no .zarr folder had a 'test' path component; "
        "treating all discovered .zarr folders as test data."
    )
    return sorted(set(all_zarr_dirs))


## 3. Safely read Zarr volume metadata

Handles both a plain `zarr.Array` store and an OME-Zarr-style `zarr.Group` (reads the first resolution array, e.g. key `"0"`). Never raises: any failure is logged and `DEFAULT_SHAPE` is used instead, so one bad store can't crash the whole pipeline.

In [ ]:
def read_zarr_shape(zarr_path: Path) -> tuple[int, ...]:
    """Safely read the array shape of a (possibly OME-Zarr) volume."""
    if zarr is None:
        print(f"[warn] zarr package not installed; using default shape for {zarr_path.name}")
        return DEFAULT_SHAPE

    try:
        store = zarr.open(str(zarr_path), mode="r")
    except Exception as exc:
        print(f"[warn] could not open {zarr_path}: {exc!r}; using default shape")
        return DEFAULT_SHAPE

    try:
        if hasattr(store, "shape"):
            shape = store.shape
        else:
            array_keys = sorted(store.array_keys())
            if not array_keys:
                raise ValueError("zarr group has no child arrays")
            shape = store[array_keys[0]].shape
    except Exception as exc:
        print(f"[warn] could not read shape for {zarr_path}: {exc!r}; using default shape")
        return DEFAULT_SHAPE

    if not shape:
        print(f"[warn] empty shape for {zarr_path}; using default shape")
        return DEFAULT_SHAPE
    return tuple(int(s) for s in shape)


def center_zyx(shape: tuple[int, ...]) -> tuple[int, int, int]:
    """Take the trailing (z, y, x) dims of `shape` and return their centers."""
    spatial = (list(DEFAULT_SHAPE[-3:]) + list(shape[-3:]))[-3:]
    return tuple(int(s) // 2 for s in spatial)


## 4. Build the dummy node/edge rows

Per dataset: 3 node rows at `t=0,1,2` positioned at the volume center, plus 2 edges linking node 0→1 and 1→2. `id` and `node_id` counters are threaded across datasets so both stay globally consecutive/unique.

In [ ]:
def build_dummy_rows(
    dataset_name: str,
    shape: tuple[int, ...],
    next_id: int,
    next_node_id: int,
) -> tuple[list[dict], int, int]:
    """3 node rows (t=0,1,2 at the volume center) + 2 edges linking them."""
    cz, cy, cx = center_zyx(shape)
    node_ids = [next_node_id, next_node_id + 1, next_node_id + 2]
    rows: list[dict] = []

    for t, node_id in zip((0, 1, 2), node_ids):
        rows.append({
            "id": next_id, "dataset": dataset_name, "row_type": "node",
            "node_id": node_id, "t": t, "z": cz, "y": cy, "x": cx,
            "source_id": -1, "target_id": -1,
        })
        next_id += 1

    for source_id, target_id in ((node_ids[0], node_ids[1]), (node_ids[1], node_ids[2])):
        rows.append({
            "id": next_id, "dataset": dataset_name, "row_type": "edge",
            "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": source_id, "target_id": target_id,
        })
        next_id += 1

    return rows, next_id, next_node_id + 3


def build_submission(zarr_dirs: Sequence[Path]) -> pd.DataFrame:
    rows: list[dict] = []
    next_id = 0
    next_node_id = 0
    for zarr_path in zarr_dirs:
        dataset_name = zarr_path.stem
        shape = read_zarr_shape(zarr_path)
        dataset_rows, next_id, next_node_id = build_dummy_rows(
            dataset_name, shape, next_id, next_node_id
        )
        rows.extend(dataset_rows)
    return pd.DataFrame(rows, columns=REQUIRED_COLUMNS)


## 5. Validate

Checks every rule from the spec: consecutive `id`, every test dataset present, node rows have `source_id=target_id=-1`, edge rows have `node_id,t,z,y,x=-1`, every `source_id`/`target_id` resolves to a real `node_id`, and there are no NaNs anywhere.

In [ ]:
def validate_submission(df: pd.DataFrame, expected_datasets: Sequence[str]) -> None:
    errors: list[str] = []

    if list(df.columns) != REQUIRED_COLUMNS:
        errors.append(f"columns mismatch: {list(df.columns)} != {REQUIRED_COLUMNS}")

    if df.isna().any().any():
        errors.append(f"NaN values found in columns: {df.columns[df.isna().any()].tolist()}")

    if df["id"].tolist() != list(range(len(df))):
        errors.append("id column is not consecutive starting at 0")

    missing = set(expected_datasets) - set(df["dataset"].unique())
    if missing:
        errors.append(f"missing datasets in submission: {sorted(missing)}")

    nodes = df[df["row_type"] == "node"]
    edges = df[df["row_type"] == "edge"]

    bad_nodes = nodes[(nodes["source_id"] != -1) | (nodes["target_id"] != -1)]
    if len(bad_nodes):
        errors.append(f"{len(bad_nodes)} node rows have source_id/target_id != -1")

    bad_edges = edges[
        (edges["node_id"] != -1) | (edges["t"] != -1) | (edges["z"] != -1)
        | (edges["y"] != -1) | (edges["x"] != -1)
    ]
    if len(bad_edges):
        errors.append(f"{len(bad_edges)} edge rows have node_id/t/z/y/x != -1")

    valid_node_ids = set(nodes["node_id"])
    bad_source = edges[~edges["source_id"].isin(valid_node_ids)]
    bad_target = edges[~edges["target_id"].isin(valid_node_ids)]
    if len(bad_source):
        errors.append(f"{len(bad_source)} edges reference an unknown source_id")
    if len(bad_target):
        errors.append(f"{len(bad_target)} edges reference an unknown target_id")

    if errors:
        raise AssertionError("Submission validation FAILED:\n- " + "\n- ".join(errors))

    n_datasets = df["dataset"].nunique()
    print(f"Validation passed: {len(df)} rows, {n_datasets} dataset(s), no issues found.")


## 6. Run the pipeline and save `submission.csv`

In [ ]:
zarr_dirs = find_test_zarr_dirs()

if not zarr_dirs:
    raise FileNotFoundError(
        "No .zarr test folders were found under /kaggle/input. "
        "Check that the competition dataset is attached to this notebook."
    )

print(f"Found {len(zarr_dirs)} test .zarr dataset(s):")
for p in zarr_dirs:
    print(f"  - {p}")

submission = build_submission(zarr_dirs)
expected_datasets = [p.stem for p in zarr_dirs]
validate_submission(submission, expected_datasets)

out_path = Path("/kaggle/working/submission.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)

print(f"\nSaved submission to {out_path}")
print(f"Shape: {submission.shape}")
submission.head(10)
